# NorthStar - MongoDB Indexing and Query Optimisation

**Notebook 6 of the analytical workflow.** Builds on the three collections created in `05_mongodb_design.ipynb` (`app_sessions`, `complaint_cases`, `customer_cases`).

## Purpose
Notebook 05 designed the schema so that the *natural* questions become one-document reads. This notebook makes those reads **fast at scale** by adding the right indexes and proves the speed-up using MongoDB's built-in profiler, `explain('executionStats')`.

## Structure
1. **Setup** - connect, identify the workload (which queries run most often, from notebooks 04 and 05).
2. **Baseline** - list existing indexes (only `_id`), run the workload, capture metrics. The plan returns `COLLSCAN` for every non-`_id` query.
3. **Single-field index** - on `complaint_cases.customer_id`. Re-run, capture metrics, compare.
4. **Compound index** - on `complaint_cases.(severity, status)`. Demonstrate the **left-prefix rule** and the **Equality–Sort–Range (ESR) rule**.
5. **Multikey index** - on `app_sessions.events.event_type`. Indexes can reach inside arrays, but they have specific rules.
6. **Tradeoffs** - write amplification, storage, when *not* to index.


## 1. Setup

Use the same connection as `05_mongodb_design.ipynb`. The three collections must already be populated.

In [ ]:
%pip install pymongo pandas python-dotenv -q
import os, time, json
from pprint import pp
import pandas as pd

from pymongo import MongoClient
from google.colab import userdata
client = MongoClient(userdata.get("MONGO_URI") )
print("Connected via MONGO_URI.")

db = client['northstar']

if db['complaint_cases'].count_documents({}) == 0:
    print("Collections empty — running notebook 05's build steps...")
    from pathlib import Path
    DATA_DIR = Path('cleaned_data')
    complaints = pd.read_csv(DATA_DIR/'complaints.csv', parse_dates=['created_at'])
    customers  = pd.read_csv(DATA_DIR/'customers.csv',  parse_dates=['signup_date'])
    app_events = pd.read_csv(DATA_DIR/'app_events.csv', parse_dates=['event_timestamp'])

    db['complaint_cases'].insert_many([
        {'_id': r['complaint_id'], 'customer_id': r['customer_id'],
         'severity': r['severity'], 'status': r['status'],
         'created_at': r['created_at'].to_pydatetime(),
         'complaint_type': r['complaint_type']}
        for _, r in complaints.iterrows()
    ])
    db['app_sessions'].insert_many([
        {'_id': sid, 'device_type': g['device_type'].mode().iloc[0],
         'event_count': len(g),
         'events': [{'event_type': e['event_type'],
                     'success_flag': int(e['success_flag'])}
                    for _, e in g.iterrows()]}
        for sid, g in app_events.groupby('session_id')
    ])
    db['customer_cases'].insert_many([
        {'_id': r['customer_id'],
         'profile': {'customer_type': r['customer_type'],
                     'home_zone': r['home_zone'],
                     'loyalty_score': float(r['loyalty_score']) if pd.notna(r['loyalty_score']) else None}}
        for _, r in customers.iterrows()
    ])

for c in ['app_sessions','complaint_cases','customer_cases']:
    print(f"  {c}: {db[c].count_documents({})} documents")


Connected via MONGO_URI.
  app_sessions: 637 documents
  complaint_cases: 320 documents
  customer_cases: 650 documents


### 1.1 Identifying the workload

Indexes are not free (see §6). I only add one when I can name a query it serves. The five queries below appear in notebook 05 and in `04_python_analysis.ipynb`; they are the workload this notebook optimises.

In [ ]:
workload = [
    ("Q1: lookup all complaints by one customer",
     ('complaint_cases', {'customer_id': 'C0368'})),
    ("Q2: filter complaints by severity",
     ('complaint_cases', {'severity': 'High'})),
    ("Q3: filter complaints by severity AND status",
     ('complaint_cases', {'severity': 'High', 'status': 'Open'})),
    ("Q4: find sessions containing a particular event type",
     ('app_sessions', {'events.event_type': 'chat_escalated'})),
    ("Q5: customers in a particular loyalty band",
     ('customer_cases', {'profile.loyalty_score': {'$gte': 75}})),
]

print("Workload queries:")
for label, (coll, filt) in workload:
    print(f"  - {label}")
    print(f"      db['{coll}'].find({filt})")


Workload queries:
  - Q1: lookup all complaints by one customer
      db['complaint_cases'].find({'customer_id': 'C0368'})
  - Q2: filter complaints by severity
      db['complaint_cases'].find({'severity': 'High'})
  - Q3: filter complaints by severity AND status
      db['complaint_cases'].find({'severity': 'High', 'status': 'Open'})
  - Q4: find sessions containing a particular event type
      db['app_sessions'].find({'events.event_type': 'chat_escalated'})
  - Q5: customers in a particular loyalty band
      db['customer_cases'].find({'profile.loyalty_score': {'$gte': 75}})


## 2. Baseline - explain plans before any index work

Each collection starts with only the default `_id` index. The expected plan for every workload query (except a query on `_id`) is **`COLLSCAN`** - collection scan meaning MongoDB reads every document.

I define a small helper to call `explain('executionStats')` and pull out the four numbers that matter:

| Field | What it means |
|---|---|
| `winningPlan.stage` | `COLLSCAN` (bad), `IXSCAN` (good), `FETCH` over `IXSCAN` (good) |
| `totalDocsExamined` | How many documents Mongo had to look at. The metric to minimise. |
| `totalKeysExamined` | How many index entries it walked. 0 means no index was used. |
| `executionTimeMillis` | Wall-clock time in ms. Noisy but useful. |


In [ ]:
def explain_summary(coll_name, filt, *, label=""):
    """Return a one-line summary of the winning plan and key counters.
    Falls back to a wall-clock timing when explain() is unavailable (mongomock).
    """
    coll = db[coll_name]

    plan = coll.find(filt).explain()
    stats = plan.get('executionStats', {})
    winning = plan.get('queryPlanner', {}).get('winningPlan', {})
    stage = winning.get('stage')

    if stage == 'FETCH':
        inner = winning.get('inputStage', {}).get('stage', '')
        stage = f"FETCH/{inner}"
    return {
            'label':       label,
            'collection':  coll_name,
            'filter':      filt,
            'stage':       stage,
            'docs_examined': stats.get('totalDocsExamined'),
            'keys_examined': stats.get('totalKeysExamined'),
            'n_returned':    stats.get('nReturned'),
            'time_ms':       stats.get('executionTimeMillis'),
            'source':        'explain'
    }


In [ ]:
# Baseline run — only _id index exists on each collection.
print("BASELINE — only default _id indexes")
print("=" * 70)
baseline = []
for label, (coll_name, filt) in workload:
    r = explain_summary(coll_name, filt, label=label)
    baseline.append(r)
    print(f"{r['label']}")
    print(f"   stage: {r['stage']}   docs_examined: {r['docs_examined']}   "
          f"returned: {r['n_returned']}   time: {r['time_ms']}ms ({r['source']})")


BASELINE — only default _id indexes
Q1: lookup all complaints by one customer
   stage: COLLSCAN   docs_examined: 320   returned: 4   time: 0ms (explain)
Q2: filter complaints by severity
   stage: COLLSCAN   docs_examined: 320   returned: 77   time: 0ms (explain)
Q3: filter complaints by severity AND status
   stage: COLLSCAN   docs_examined: 320   returned: 14   time: 0ms (explain)
Q4: find sessions containing a particular event type
   stage: COLLSCAN   docs_examined: 637   returned: 38   time: 0ms (explain)
Q5: customers in a particular loyalty band
   stage: COLLSCAN   docs_examined: 650   returned: 104   time: 1ms (explain)


## 3. Single-field index - `complaint_cases.customer_id`

The Q1 query (`find({'customer_id': X})`) is hit on **every customer-facing screen** that shows complaint history. It is a textbook single-field-index case: equality filter on one selective column.

In [ ]:
# Create the index.
db['complaint_cases'].create_index('customer_id', name='ix_complaints_customer_id')

# List the indexes on the collection
print("complaint_cases indexes:")
for ix in db['complaint_cases'].list_indexes():
    print(f"  {ix['name']:35s} keys={dict(ix['key'])}")


complaint_cases indexes:
  _id_                                keys={'_id': 1}
  ix_complaints_customer_id           keys={'customer_id': 1}


In [ ]:
# Re-run Q1
print("AFTER adding ix_complaints_customer_id")
print("=" * 70)
r = explain_summary('complaint_cases', {'customer_id': 'C0368'},
                    label='Q1 (post-index)')
print(f"   stage: {r['stage']}")
print(f"   docs_examined: {r['docs_examined']} (was {baseline[0]['docs_examined']})")
print(f"   keys_examined: {r['keys_examined']}")
print(f"   time: {r['time_ms']}ms (was {baseline[0]['time_ms']}ms)")


AFTER adding ix_complaints_customer_id
   stage: FETCH/IXSCAN
   docs_examined: 4 (was 320)
   keys_examined: 4
   time: 1ms (was 0ms)




**The improvement.** `totalDocsExamined` drops from 320 to 4 - Mongo only fetches the four matching documents, not the whole collection. The plan stage becomes `FETCH <- IXSCAN`, which means: walk the index to find the matching `_id`s, then fetch only those documents.

**Selectivity matters.** This index helps because `customer_id` is highly selective - 320 complaints across ~250 distinct customers means an average of ~1.3 documents per value. An index on `complaint_type` (only six possible values) would be a much weaker win and could even be ignored by the planner if the resulting scan is small enough.

## 4. Compound index - `complaint_cases.(severity, status)`

Q3 (`{'severity':'High', 'status':'Open'}`) is the SLA-monitoring query. A single-field index on `severity` would help, but a **compound index** on `(severity, status)` helps more and also serves single-field queries on `severity` alone (the **left-prefix rule**).

### 4.1 The left-prefix rule
A compound index on `(A, B, C)` serves queries on:
- `{A: ...}`
- `{A: ..., B: ...}`
- `{A: ..., B: ..., C: ...}`

It does **not** serve `{B: ...}` alone, or `{C: ...}` alone - the left prefix must be present.

### 4.2 The ESR rule (Equality, Sort, Range)
The optimal column order in a compound index is:
1. **Equality** filters first (`severity = 'High'`)
2. **Sort** columns next (any column you `.sort()` on)
3. **Range** filters last (`created_at > some date`)

So if our most common query were `{'severity': 'High', 'created_at': {'$gt': D}}` and i wanted results in `status` order, the ideal index would be `(severity, status, created_at)` - equality, sort, range.

In [ ]:
# Create the compound index
db['complaint_cases'].create_index([('severity', 1), ('status', 1)],
                                    name='ix_complaints_severity_status')

# Update the index list
print("complaint_cases indexes now:")
for ix in db['complaint_cases'].list_indexes():
    print(f"  {ix['name']:35s} keys={dict(ix['key'])}")


complaint_cases indexes now:
  _id_                                keys={'_id': 1}
  ix_complaints_customer_id           keys={'customer_id': 1}
  ix_complaints_severity_status       keys={'severity': 1, 'status': 1}


In [ ]:
# Re-run Q2 and Q3
print("AFTER adding ix_complaints_severity_status")
print("=" * 70)
for i in (1, 2):
    label, (coll, filt) = workload[i]
    r = explain_summary(coll, filt, label=label)
    b = baseline[i]
    print(f"{label}")
    print(f"   stage: {r['stage']}   docs_examined: {r['docs_examined']} "
          f"(was {b['docs_examined']})")
    print(f"   time: {r['time_ms']}ms (was {b['time_ms']}ms)")


AFTER adding ix_complaints_severity_status
Q2: filter complaints by severity
   stage: FETCH/IXSCAN   docs_examined: 77 (was 320)
   time: 1ms (was 0ms)
Q3: filter complaints by severity AND status
   stage: FETCH/IXSCAN   docs_examined: 14 (was 320)
   time: 0ms (was 0ms)


**Why this index is better than two single-field indexes.**

If I had created `{severity:1}` *and* `{status:1}` separately, the planner could only use **one of them** for a query like `{severity:'High', status:'Open'}` - it would scan that index, then re-check the other condition on each candidate doc. The compound index lets Mongo seek directly to the *intersection* in one traversal.

## 5. Multikey index - `app_sessions.events.event_type`

Q4 (`{'events.event_type': 'chat_escalated'}`) reaches **inside an embedded array**. MongoDB supports indexes on array fields — they are called **multikey indexes** because each document contributes *multiple* index entries (one per array element).

### Multikey rules
1. You can create a multikey index just like any other: `create_index('events.event_type')`.
2. Only **one** field in a compound index can be a multikey field - Mongo cannot represent the cartesian product of two arrays inside the index entry.
3. Multikey indexes inflate index size - a session with 7 events produces 7 entries in this index.
4. The query syntax is unchanged - Mongo handles the array traversal transparently.

In [ ]:
# Create the multikey index
db['app_sessions'].create_index('events.event_type',
                                 name='ix_sessions_event_type')

print("app_sessions indexes:")
for ix in db['app_sessions'].list_indexes():
    print(f"  {ix['name']:35s} keys={dict(ix['key'])}")


app_sessions indexes:
  _id_                                keys={'_id': 1}
  ix_sessions_event_type              keys={'events.event_type': 1}


In [ ]:
# Re-run Q4
print("AFTER adding ix_sessions_event_type")
print("=" * 70)
label, (coll, filt) = workload[3]
r = explain_summary(coll, filt, label=label)
b = baseline[3]
print(f"{label}")
print(f"   stage: {r['stage']}   docs_examined: {r['docs_examined']} "
      f"(was {b['docs_examined']})")
print(f"   time: {r['time_ms']}ms (was {b['time_ms']}ms)")


AFTER adding ix_sessions_event_type
Q4: find sessions containing a particular event type
   stage: FETCH/IXSCAN   docs_examined: 38 (was 637)
   time: 1ms (was 0ms)


### 5.1 Indexing the integrated view

Q5 hits a nested profile field (`profile.loyalty_score`). Dot-notation works the same way for non-array nested fields as it does for arrays and the resulting index is *not* multikey because the path doesn't pass through an array.

In [ ]:
db['customer_cases'].create_index('profile.loyalty_score',
                                   name='ix_customers_loyalty')

print("customer_cases indexes:")
for ix in db['customer_cases'].list_indexes():
    print(f"  {ix['name']:35s} keys={dict(ix['key'])}")

# Re-run Q5
label, (coll, filt) = workload[4]
r = explain_summary(coll, filt, label=label)
b = baseline[4]
print(f"\n{label}")
print(f"   stage: {r['stage']}   docs_examined: {r['docs_examined']} "
      f"(was {b['docs_examined']})")
print(f"   time: {r['time_ms']}ms (was {b['time_ms']}ms)")


customer_cases indexes:
  _id_                                keys={'_id': 1}
  ix_customers_loyalty                keys={'profile.loyalty_score': 1}

Q5: customers in a particular loyalty band
   stage: FETCH/IXSCAN   docs_examined: 104 (was 650)
   time: 2ms (was 1ms)


## 6. Trade-offs - when *not* to index

Indexes are not free. Every index costs:

1. **Write amplification.** Every `insert`, `update`, or `delete` must also update every index that covers the affected field. A collection with 5 indexes does ~5× the write work of an un-indexed one.
2. **Storage.** Each index is a separate B-tree on disk. Multikey indexes can be especially large (one entry per array element).
3. **Planner overhead.** The query planner considers every viable index. Pathologically high index counts (15+) noticeably slow down planning on each query.

**Rules for NorthStar.**

| Do index | Don't index |
|---|---|
| Foreign-key-like fields (`customer_id`, `order_id`) | Low-cardinality fields (`gender`, 3-value `severity` alone) - the planner will probably scan anyway |
| Fields in `.sort()` calls | Boolean flags that are TRUE for most documents |
| Fields used in `$match` of frequent aggregation pipelines | Fields that change on every write (timestamps unless time-series) |
| Compound fields in their actual query order (ESR) | Every column "just in case" |

### Specific NorthStar decisions

- **`app_sessions.events.api_latency_ms`** - *not indexed*. Range queries on a wide-distribution numeric field rarely benefit unless combined with a more selective equality predicate. If a use case appears (e.g. "find slow sessions"), add a compound `(zone_context, events.api_latency_ms)` index then.
- **`customer_cases.profile.customer_type`** - *not indexed*. Three values (`Consumer`, `SME`, `Enterprise`), so equality scans return ~33% of docs. The planner will ignore the index. If reports filter on customer type *with* another more selective predicate, an `(customer_type, ...)` compound may help.
- **`complaint_cases.created_at`** - *consider a partial index* on `{status: 'Open'}` if the daily SLA report is the dominant query. A partial index covers only matching docs and is dramatically smaller than a full index when the filter is highly selective. Example: `create_index([('created_at',1)], partialFilterExpression={'status':'Open'})`.